# Triangle cavity prediction: collect measurements near the transition

Notebook 06 already gives a broad comparison at three sizes, with 60 independent
trials at each parameter. We keep those measurements. This notebook adds only
triangle graphs on a closer grid around the cavity prediction from notebook 08,
$\lambda_{\triangle}\approx1.22860$. The extra size $n=30{,}000$ helps check
whether the finite-size rise moves toward that prediction.

Each trial uses the same triangle sampler and simple-graph convention as before.
The membership parameter is still $\lambda$, not the mean graph degree
$2\lambda$. No configuration graphs are regenerated. Open notebook 10 afterwards
to compare these new measurements with the saved broad study and both predictions.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/02_algorithms.ipynb"

PROJECT_ROOT = NOTEBOOK_DIR.parent

In [ ]:
import csv
from numbers import Integral
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed

## Choose a focused experiment

The grid is closest around $1.2286$, with outer points to see the tails of the
finite-size transition. It supplements the broad grid rather than repeating it.
The rounded value $1.2286$ is slightly above the predicted threshold; it is not
an experiment at exactly the mathematical tangency.

The default run has 4,000 independent graphs. Trials run in parallel within one
size, and each size is saved before the next starts. `jobs=1` is useful for a
serial check, positive values select the worker count, and `-1` uses all CPUs.
Changing the worker count preserves every trial's seed and measurement.

Choose another output directory to keep a second study. Existing CSVs are
protected unless you explicitly set `overwrite=True`. Notebook 10 needs the
same directory under `refinement_dir`.

In [ ]:
config = {
    "sizes": [1000, 3000, 10000, 30000],  # Extend the saved study by one larger size.
    "lambdas": [1.19, 1.21, 1.22, 1.225, 1.2286, 1.2325, 1.2375, 1.245, 1.255, 1.27],
    "repeats": 100,  # Independent graphs at each size and parameter value.
    "seed": 20261110,  # Separate from notebook 06; each CSV row saves its replay seed.
    "jobs": 4,  # Parallel independent trials, with no nested worker pool.
    "output_dir": "results/runs/triangle_cavity",
    "overwrite": False,  # Protect measurements unless replacement is intentional.
}
config

## Generate and measure the same triangle model

We reuse `construct_special_triangle_RIG` and the tested adjacency-list peeling
algorithm from notebook 02. Each graph is generated with an explicit NumPy
random stream. The core fraction includes isolated vertices in its denominator.
UUID labels change on replay; the sampled structure and core fraction do not.

The CSV uses the same seven core-measurement columns as notebook 03. The
additional broad-study diagnostics, such as degree histograms and transitivity,
already exist in notebook 06's saved data and need not be collected again here.

All seeds are assigned before dispatch. A size has its own stream, independent
of size-list ordering. The lambda grid and repetition count determine the order
of trials in that stream, so changing either can change their individual seeds.

In [ ]:
REFINEMENT_FIELDS = (
    "model",
    "n",
    "lambda",
    "replicate",
    "seed",
    "core_size",
    "core_fraction",
)


def refinement_trial(n: int, lam: float, replicate: int, seed: int) -> dict:
    """Sample one triangle graph and measure its simple-graph 3-core."""
    if not isinstance(n, Integral) or isinstance(n, bool) or n <= 0:
        raise ValueError("n must be a positive integer")
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("lambda must be finite and nonnegative")
    graph = construct_special_triangle_RIG(n, lam, rng=np.random.default_rng(seed))
    core_size = int(core_mask(graph_adjacency(graph), 3).sum())
    return {
        "model": "triangle_rig",
        "n": int(n),
        "lambda": float(lam),
        "replicate": int(replicate),
        "seed": int(seed),
        "core_size": core_size,
        "core_fraction": core_size / n,
    }


def simulate_refinement(n, lambdas, repeats, seed=20261110, jobs=4) -> list[dict]:
    """Run a seeded parameter grid without depending on worker scheduling."""
    for name, value in (("n", n), ("repeats", repeats), ("seed", seed), ("jobs", jobs)):
        if not isinstance(value, Integral) or isinstance(value, bool):
            raise ValueError(f"{name} must be an integer")
    if n <= 0 or repeats <= 0 or seed < 0 or jobs == 0:
        raise ValueError("Use positive n/repeats, a nonnegative seed, and nonzero jobs")
    grid = [float(lam) for lam in lambdas]
    if (
        not grid
        or len(set(grid)) != len(grid)
        or any(not np.isfinite(lam) or lam < 0 for lam in grid)
    ):
        raise ValueError("Use distinct, finite, nonnegative lambda values")
    children = iter(
        np.random.SeedSequence([int(seed), int(n)]).spawn(len(grid) * repeats)
    )
    tasks = [
        (n, lam, replicate, int(next(children).generate_state(1, dtype=np.uint64)[0]))
        for lam in grid
        for replicate in range(repeats)
    ]
    return Parallel(n_jobs=jobs, verbose=5)(
        delayed(refinement_trial)(*task) for task in tasks
    )


def run_refinement(config: dict, project_root: str | Path = PROJECT_ROOT) -> list[Path]:
    """Preflight all targets, then save each completed size as its own CSV."""
    sizes = list(config["sizes"])
    if (
        not sizes
        or len(set(sizes)) != len(sizes)
        or any(
            not isinstance(n, Integral) or isinstance(n, bool) or n <= 0 for n in sizes
        )
    ):
        raise ValueError("sizes must contain distinct positive integers")
    directory = Path(project_root) / config["output_dir"]
    paths = [directory / f"cavity_refinement_n{n}.csv" for n in sizes]
    overwrite = config.get("overwrite", False)
    existing = [str(path) for path in paths if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(
            "Measurements already exist: "
            + ", ".join(existing)
            + ". Choose another output_dir or explicitly set overwrite=True."
        )
    total = len(sizes) * len(config["lambdas"]) * config["repeats"]
    print(f"Triangle refinement: {total:,} independent graphs across sizes {sizes}.")
    for n, path in zip(sizes, paths):
        started = perf_counter()
        rows = simulate_refinement(
            n, config["lambdas"], config["repeats"], config["seed"], config["jobs"]
        )
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open(
            "w" if overwrite else "x", newline="", encoding="utf-8"
        ) as stream:
            writer = csv.DictWriter(stream, fieldnames=REFINEMENT_FIELDS)
            writer.writeheader()
            writer.writerows(rows)
        print(
            f"n={n:,}: saved {len(rows):,} trials to {path} in {perf_counter() - started:.1f} s."
        )
    return paths

## Run the additional trials

This is the computational part. Its output consists only of new measurement
CSVs under `results/runs/triangle_cavity/`; the broad-study data is read later
and remains available separately. To regenerate figures from existing data,
run notebook 10 directly or use `make triangle-figures`.

In [ ]:
data_files = run_refinement(config)
data_files

## Continue with the scientific comparison

Notebook 10 combines the existing broad study with this refinement, retaining
the source of each measurement. It examines the mean, the middle half of graph
outcomes, the frequency of substantial cores, and individual distributions near
the predicted transition. Refinement has more repetitions and a different grid,
so error summaries keep the two experimental designs separate.